# Trying to break `# @cash:cache-calls`

A copy of the demo notebook, turned adversarial. Seventeen attacks; **one found
a real bug**, which is reproduced live at the bottom.

Everything here is checked against *ground truth* — what plain Python would do —
not against what the badge says. A cache that agrees with itself proves nothing.

| # | attack | result |
|---|---|---|
| 1 | callee reads a global that then changes | ✅ correct |
| 2 | caller mutates a value returned from cache | ✅ correct |
| 3 | callee's source edited between runs | ✅ correct |
| 4 | short-circuited call (`f() or g()`) | ✅ never runs `g` |
| 5 | exception inside a cached call | ✅ raises every time |
| 6 | unseeded randomness inside a cached call | ✅ frozen **and warned** |
| 7 | nested `f(g(x))` | ✅ correct (`g` re-runs; see below) |
| 8 | lambda as the callee | ✅ correct |
| 9 | walrus in the arguments | ✅ correct |
| 10 | callee rebound to a new function, same name | ✅ correct |
| 11 | kernel restart | ✅ cache survives |
| 12 | helper defined in a *later* cell, then edited | ✅ correct |
| 13 | **Figure + bare `plt.savefig()`** | ❌ **BUG — fixed** |

Run the cells below to reproduce any of them yourself.

In [ ]:
import cash
%cash_on
%cash_badge print          # text badges: easier to read than the HTML ones here

## The one that broke it

`statement/processor.py` refuses to cache a matplotlib Figure. The RAM tier
deep-copies on store, and `Figure.__setstate__` re-registers **the copy** as
pyplot's *current figure* — so a later bare `plt.savefig()` writes the cache's
snapshot instead of the figure you drew on.

Routing calls through the decorator skipped that guard.

Two details make this hard to spot:

1. It bites on the **first** run, during the *store* — not on a later cache hit.
   "Nothing is cached yet, so the first run must be fine" sends you to the wrong
   place.
2. It only appears when *creation* and *drawing* are separate statements. A
   function that builds and draws in one go cannot expose it.

In [ ]:
import time, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib, tempfile

TMP = pathlib.Path(tempfile.mkdtemp())

def new_fig(n):
    time.sleep(0.2)              # above the cost-model floor, or nothing stores
    fig, ax = plt.subplots()
    return fig, ax

def check(label, fig, ax):
    """Save the SAME figure twice - through the object, and through pyplot."""
    o, g = TMP / f"{label}_obj.png", TMP / f"{label}_glb.png"
    fig.savefig(o); plt.savefig(g)
    same = o.read_bytes() == g.read_bytes()
    print(f"{label:<22} object={o.stat().st_size}B  pyplot={g.stat().st_size}B  "
          f"identical={same}  plt.gcf() is fig -> {plt.gcf() is fig}")
    return same

### Ground truth — no caching at all

Both saves must be byte-identical, and `plt.gcf()` must be your figure.

In [ ]:
%cash_off
holder = []
for n in [1]:
    holder.append(new_fig(n))
fig, ax = holder[-1]
ax.bar(['a', 'b'], [3, 6])
check("no caching", fig, ax)
%cash_on

### With `# @cash:cache-calls` — fixed

`CallCache` now passes a `cache_if` predicate that refuses identity-coupled
results *before* the write, so the damaging deep copy is never made.

In [ ]:
holder2 = []
# @cash:cache-calls
for n in [2]:
    holder2.append(new_fig(n))
fig2, ax2 = holder2[-1]
ax2.bar(['a', 'b'], [3, 6])
check("cache-calls", fig2, ax2)

### The decorator, by hand — **still broken (CAS-245)**

The same defect reproduces when *you* write `@cash.cache`, so it predates this
feature: `cache-calls` only made it reachable without anyone choosing to
decorate a plotting function.

This cell is a live reproduction of an open bug. Expect `identical=False` and
`plt.gcf() is fig -> False`, and note the two file sizes differ — the pyplot one
is a figure with no bars on it.

In [ ]:
_c = cash.Cash()

@_c.cache
def new_fig_decorated(n):
    time.sleep(0.2)
    fig, ax = plt.subplots()
    return fig, ax

fig3, ax3 = new_fig_decorated(3)
ax3.bar(['a', 'b'], [3, 6])
check("hand-decorated", fig3, ax3)     # <-- expected to FAIL: this is CAS-245

## Two behaviours that are correct but surprising

Neither is a bug; both are worth knowing before you rely on the directive.

### Nested calls: the inner one still runs

In `out.append(f(g(x)))` only `f` is intercepted. `g(x)` has to be evaluated on
every pass, because its **value** is what keys `f`.

So put the expensive work in the *outer* call. If `g` is the slow one, this
directive will not help you.

In [ ]:
INNER = []
def g(x):
    INNER.append(x)
    return x
def f(y):
    time.sleep(0.2)
    return y + 1

out = []
# @cash:cache-calls
for x in [1, 2]:
    out.append(f(g(x)))
print("INNER", INNER, "-> run this cell again and watch it grow")

### Randomness is frozen — but here it is *louder* than usual

A cached unseeded draw is replayed, by design. Interception routes through the
decorator's randomness gate, which **warns**. The same draw under ordinary
statement caching is frozen *silently* — so on this axis the directive makes
things more visible, not less.

In [ ]:
import random
def draw(x):
    time.sleep(0.2)
    return random.random()

vals = []
# @cash:cache-calls
for x in [1]:
    vals.append(draw(x))
print("VALS", vals, "- re-run: identical, and note the CashRandomnessWarning")

## What this did not test

Being honest about the edges of the sweep:

- **Concurrency.** Nothing here runs two kernels or threads against one cache.
- **Large arguments.** `compute(big_df, x)` pickles the frame on every call;
  that is a known cost concern, not measured here.
- **Bound methods.** Deliberately never intercepted — see the annotations docs
  for why.
- **Impure functions in general.** A cached call skips its side effects. That is
  documented, and cash warns, but the warning is advisory: it does not stop the
  cache. If your function writes files or mutates globals, read the warning.